# 04_Metrics.ipynb: Prospective Replay & Rigorous Acceptance Evaluation
Tuân thủ nghiêm ngặt AGENTS.md và PROCESS.md: Không rò rỉ dữ liệu, đánh giá holdout hoàn toàn đóng băng (frozen model, frozen policy, frozen protocol), không fit/refit, không chọn lại mô hình hay ngưỡng trên holdout. Mọi quyết định cảnh báo trực tuyến là nhân quả (causal) và prequential.

In [1]:
# 1. Environment & Artifact Integrity Verification
get_ipython().run_line_magic('run', '02_Preprocessing.ipynb')

import sys
import hashlib
import joblib
import numpy as np
import pandas as pd
import sklearn

print('Python version :', sys.version.split()[0])
print('NumPy version  :', np.__version__)
print('Pandas version :', pd.__version__)
print('Scikit-Learn   :', sklearn.__version__)
print('Joblib version :', joblib.__version__)

exp = joblib.load('experiment.joblib')

h_secom = hashlib.sha256(open('data/secom/secom.data', 'rb').read()).hexdigest()
h_labels = hashlib.sha256(open('data/secom/secom_labels.data', 'rb').read()).hexdigest()

assert h_secom == exp['file_hashes']['secom_data_sha256'], 'SECOM raw data SHA256 mismatch!'
assert h_labels == exp['file_hashes']['secom_labels_sha256'], 'SECOM labels SHA256 mismatch!'
print('Independent SHA256 verification passed successfully.')
print('SECOM Data SHA256  :', h_secom)
print('SECOM Labels SHA256:', h_labels)

All synthetic tests and causal checks passed.
df_dev shape: (1253, 593)
X_dev shape: (269, 15), meta_dev shape: (269, 9)
Event label counts:
event_label
0.0    157
1.0     72
NaN     40
Name: count, dtype: int64
Maturity mask sum at fit_time: 17 / 269
capacity_threshold_at_origin future-invariance test passed.


Python version : 3.13.1
NumPy version  : 2.5.4
Pandas version : 3.0.6
Scikit-Learn   : 1.9.1
Joblib version : 1.6.0


Independent SHA256 verification passed successfully.
SECOM Data SHA256  : 20f0e7ee434f7dcbae0eea9ffff009a2b57f42d6b0dc9a5bd4f00782c0a3374c
SECOM Labels SHA256: 126884cf453705c9e61a903fe906f0665a3b45ce3639e621edc5c93c89627e03


In [2]:
# 2. Frozen Protocol, Feature Build & Observation Accounting
lead_delta = pd.Timedelta(exp['protocol']['lead'])
horizon_delta = pd.Timedelta(exp['protocol']['horizon'])
history_delta = pd.Timedelta(exp['protocol']['history'])
delay_delta = pd.Timedelta(exp['protocol']['delay'])
cadence_delta = pd.Timedelta(exp['protocol']['cadence'])
q_val = float(exp['protocol']['q'])
min_event_samples = int(exp['protocol']['min_event_samples'])

dev_cutoff = pd.Timestamp(exp['protocol']['dev_cutoff'])
model_ready_time = pd.Timestamp(exp['policy']['model_ready_time'])
policy_ready_time = pd.Timestamp(exp['policy']['policy_ready_time'])
earliest_holdout_origin = pd.Timestamp(exp['temporal_bounds']['earliest_holdout_origin'])
planned_eval_start = pd.Timestamp(exp['temporal_bounds']['planned_eval_start'])
planned_eval_end = pd.Timestamp(exp['temporal_bounds']['planned_eval_end'])

print('=== FROZEN TEMPORAL BOUNDS ===')
print('Model Ready Time       :', model_ready_time)
print('Policy Ready Time      :', policy_ready_time)
print('Earliest Holdout Origin:', earliest_holdout_origin)
print('Planned Eval Span      : [', planned_eval_start, ',', planned_eval_end, ')')

df_raw = load_secom()
X_full, meta_full = build_forecast_dataset(
    df_raw,
    lead=lead_delta,
    horizon=horizon_delta,
    history=history_delta,
    delay=delay_delta,
    cadence=cadence_delta,
    q=q_val,
    min_event_samples=min_event_samples,
    include_sensor_means=True
)

assert (meta_full['max_feature_available'] <= meta_full.index).all(), 'Future leakage: max_feature_available > origin!'
eval_origin_mask = (
    (meta_full.index >= earliest_holdout_origin) &
    (meta_full['target_start'] >= planned_eval_start) &
    (meta_full['target_end'] <= planned_eval_end)
)
eval_origins = meta_full.index[eval_origin_mask]
assert len(eval_origins) > 0, 'No holdout origins found!'
assert eval_origins.is_unique, 'Eval origins index must be unique!'
assert (meta_full.loc[eval_origins, 'target_start'] >= planned_eval_start).all(), 'Target start precedes planned eval start!'
assert (meta_full.loc[eval_origins, 'target_end'] <= planned_eval_end).all(), 'Target end exceeds planned eval end!'
assert policy_ready_time <= eval_origins.min(), 'Policy ready time exceeds first eval origin!'
assert model_ready_time <= policy_ready_time, 'Model ready time exceeds policy ready time!'

# Accounting across entire holdout cutoff (known 314 records)
holdout_raw = df_raw[df_raw['timestamp'] >= dev_cutoff]
n_holdout_total = len(holdout_raw)
fail_holdout_total = int(holdout_raw['fail'].sum())

pre_span = df_raw[(df_raw['timestamp'] >= dev_cutoff) & (df_raw['timestamp'] < planned_eval_start)]
in_span = df_raw[(df_raw['timestamp'] >= planned_eval_start) & (df_raw['timestamp'] < planned_eval_end)]
post_span = df_raw[df_raw['timestamp'] >= planned_eval_end]

print('\n=== RAW HOLDOUT OBSERVATION ACCOUNTING ===')
print('Total Holdout (timestamp >= dev_cutoff) :', n_holdout_total, 'records,', fail_holdout_total, 'Fails (historical non-blind 314)')
print('  - Pre-evaluation warmup span          :', len(pre_span), 'records,', int(pre_span['fail'].sum()), 'Fails')
print('  - Planned evaluation span             :', len(in_span), 'records,', int(in_span['fail'].sum()), 'Fails')
print('  - Post-evaluation remainder span      :', len(post_span), 'records,', int(post_span['fail'].sum()), 'Fails')
assert len(pre_span) + len(in_span) + len(post_span) == n_holdout_total, 'Holdout partition integrity mismatch!' 

=== FROZEN TEMPORAL BOUNDS ===
Model Ready Time       : 2008-09-17 12:00:00
Policy Ready Time      : 2008-10-02 20:54:00
Earliest Holdout Origin: 2008-10-03 00:00:00
Planned Eval Span      : [ 2008-10-03 06:00:00 , 2008-10-17 06:00:00 )



=== RAW HOLDOUT OBSERVATION ACCOUNTING ===
Total Holdout (timestamp >= dev_cutoff) : 314 records, 17 Fails (historical non-blind 314)
  - Pre-evaluation warmup span          : 15 records, 1 Fails
  - Planned evaluation span             : 297 records, 16 Fails
  - Post-evaluation remainder span      : 2 records, 0 Fails


In [3]:
# 3. Prequential Scoring across all Contexts (Continuous from model_ready_time)
scoring_mask = (meta_full.index >= model_ready_time) & (meta_full.index <= eval_origins.max())
meta_scoring = meta_full.loc[scoring_mask]
X_scoring = X_full.loc[scoring_mask]

assert (meta_scoring.index >= model_ready_time).all(), 'Scoring index precedes model_ready_time!'
assert meta_scoring.index.is_unique, 'meta_scoring index must be unique!'

def predict_score(name, model_obj, X_mat):
    if isinstance(model_obj, dict):
        mtype = model_obj.get('type')
        if mtype == 'never_alert':
            return pd.Series(-np.inf, index=X_mat.index)
        elif mtype == 'baseline':
            col = model_obj['col']
            return X_mat[col].fillna(0.0).astype(float)
        elif mtype == 'prior_fallback':
            return pd.Series(float(model_obj['value']), index=X_mat.index)
        else:
            raise ValueError('Unknown dict model type: ' + str(mtype))
    elif isinstance(model_obj, sklearn.pipeline.Pipeline):
        feats = list(model_obj.feature_names_in_)
        X_sub = X_mat[feats]
        last_step = model_obj.steps[-1][1]
        if hasattr(last_step, 'predict_proba'):
            probs = model_obj.predict_proba(X_sub)[:, 1]
            return pd.Series(probs, index=X_mat.index)
        elif hasattr(last_step, 'predict'):
            preds = np.clip(model_obj.predict(X_sub), 0.0, 1.0)
            return pd.Series(preds, index=X_mat.index)
        else:
            raise TypeError('Pipeline step has neither predict_proba nor predict')
    else:
        raise TypeError('Unrecognized model object type: ' + str(type(model_obj)))

print('Scoring contexts available:', len(meta_scoring), 'origins from', meta_scoring.index.min(), 'to', meta_scoring.index.max())

Scoring contexts available: 115 origins from 2008-09-17 12:00:00 to 2008-10-16 00:00:00


In [4]:
# 4. Prospective Policy Replay & Comparative Benchmarking
lookback_delta = pd.Timedelta(exp['policy']['lookback'])
n_quantiles_policy = int(exp['policy']['n_quantiles'])
frozen_thresholds = exp['frozen_thresholds']
deploy_models = exp['deploy_models']

winner_info = exp['winner']
selected_name = winner_info['candidate']
selected_budget = float(winner_info['budget'])
print('Primary Selected Policy : candidate =', selected_name, ', budget =', selected_budget)

# Selected development diagnostics from artifact diagnostics
if 'diagnostics' in exp:
    print('\n=== SELECTED DEVELOPMENT DIAGNOSTICS (FROM ARTIFACT) ===')
    for f_k in sorted(exp['diagnostics'].keys()):
        f_diag = exp['diagnostics'][f_k].get(selected_name, {})
        tr_ap = f_diag.get('train_ap', np.nan)
        va_ap = f_diag.get('val_ap', np.nan)
        print(f'{f_k} | train_ap: {tr_ap:.4f} | val_ap: {va_ap:.4f}')

eval_meta = meta_full.loc[eval_origins]
eval_X = X_full.loc[eval_origins]

def run_policy_replay(cand_name, budget_val):
    m_obj = deploy_models[cand_name]
    scores_series = predict_score(cand_name, m_obj, X_scoring)
    init_tau = frozen_thresholds[cand_name][budget_val]

    taus = {}
    audits = {}
    warns = []

    for orig in eval_origins:
        tau, audit = capacity_threshold_at_origin(
            scores=scores_series,
            metadata=meta_scoring,
            raw_df=df_raw,
            current_origin=orig,
            model_ready_time=model_ready_time,
            budget=budget_val,
            lookback=lookback_delta,
            min_lead=lead_delta,
            n_quantiles=n_quantiles_policy,
            frozen_initial_threshold=init_tau
        )
        if pd.notna(audit['last_label_available_time']):
            assert audit['last_label_available_time'] <= orig, 'Label leak at ' + str(orig)
        if pd.notna(audit['past_target_end_max']):
            assert audit['past_target_end_max'] <= orig - delay_delta, 'Target leak at ' + str(orig)

        sc = scores_series.loc[orig]
        w = bool(sc >= tau)
        taus[orig] = tau
        audits[orig] = audit
        warns.append(w)

    warn_series = pd.Series(warns, index=eval_origins)
    metrics = interval_metrics(
        df_raw=df_raw,
        metadata=eval_meta,
        warnings_bool=warn_series.values,
        eval_start=planned_eval_start,
        eval_end=planned_eval_end,
        min_lead=lead_delta
    )
    return {
        'candidate': cand_name,
        'budget': budget_val,
        'metrics': metrics,
        'warn_series': warn_series,
        'taus': pd.Series(taus),
        'audits': audits,
        'scores': scores_series.loc[eval_origins]
    }

benchmark_candidates = [selected_name, 'never_alert', 'baseline_rate24h', 'baseline_rate72h', 'baseline_rate168h']
seen_configs = set()
benchmark_results = []
replays_dict = {}

for c_name in benchmark_candidates:
    for b_val in [0.20, 0.25, 0.30]:
        cfg_key = (c_name, b_val)
        if cfg_key in seen_configs:
            continue
        seen_configs.add(cfg_key)
        res = run_policy_replay(c_name, b_val)
        replays_dict[cfg_key] = res
        m = res['metrics']
        benchmark_results.append({
            'candidate': c_name,
            'budget': b_val,
            'covered_fail': m['covered_fail'],
            'total_fail': m['total_fail'],
            'coverage': m['coverage'],
            'covered_n': m['covered_n'],
            'total_n': m['total_n'],
            'workload': m['workload'],
            'time_under_alert': m['time_under_alert'],
            'event_precision': m['event_precision'],
            'event_recall': m['event_recall']
        })

df_comparison = pd.DataFrame(benchmark_results)
selected_replay = replays_dict[(selected_name, selected_budget)]
print('\n=== COMPARATIVE POLICY BENCHMARKING ON PLANNED EVAL SPAN ===')
print(df_comparison.sort_values(by=['candidate', 'budget']).to_string(index=False))

Primary Selected Policy : candidate = lr_history_c01 , budget = 0.2

=== SELECTED DEVELOPMENT DIAGNOSTICS (FROM ARTIFACT) ===
fold_1 | train_ap: 0.8512 | val_ap: 0.1633
fold_2 | train_ap: 0.8134 | val_ap: 0.5302
fold_3 | train_ap: 0.7242 | val_ap: 0.1385


C:\Users\ductr\AppData\Local\Temp\ipykernel_10832\352093625.py:2: Pandas4Warning: 'd' is deprecated and will be removed in a future version. Please use 'D' instead of 'd'.
  lookback_delta = pd.Timedelta(exp['policy']['lookback'])



=== COMPARATIVE POLICY BENCHMARKING ON PLANNED EVAL SPAN ===
        candidate  budget  covered_fail  total_fail  coverage  covered_n  total_n  workload  time_under_alert  event_precision  event_recall
baseline_rate168h    0.20            14          16    0.8750        173      297  0.582492          0.446429         0.333333      0.636364
baseline_rate168h    0.25            15          16    0.9375        177      297  0.595960          0.482143         0.347826      0.727273
baseline_rate168h    0.30            15          16    0.9375        201      297  0.676768          0.660714         0.333333      0.909091
 baseline_rate24h    0.20             4          16    0.2500         78      297  0.262626          0.214286         0.000000      0.000000
 baseline_rate24h    0.25             4          16    0.2500        110      297  0.370370          0.339286         0.000000      0.000000
 baseline_rate24h    0.30             7          16    0.4375        128      297  0.430976 

In [5]:
# 5. Primary Table Diagnostics & Complete Per-Day Breakdown
selected_metrics = selected_replay['metrics']
cov_n = selected_metrics['covered_n']
cov_fail = selected_metrics['covered_fail']
cov_pass = cov_n - cov_fail
pass_fraction = (cov_pass / cov_n) if cov_n > 0 else np.nan

sel_warns = selected_replay['warn_series']
high_origins = int(sel_warns.sum())
valid_ev = ~eval_meta['event_label'].isna()
false_high_events = int(((sel_warns == 1) & (eval_meta['event_label'] == 0) & valid_ev).sum())
high_sparse_origins = int(((sel_warns == 1) & eval_meta['event_label'].isna()).sum())

primary_summary = pd.DataFrame([{
    'policy': selected_name + ' @ ' + str(selected_budget),
    'covered_fail': cov_fail,
    'total_fail': selected_metrics['total_fail'],
    'coverage': selected_metrics['coverage'],
    'covered_n': cov_n,
    'total_n': selected_metrics['total_n'],
    'workload': selected_metrics['workload'],
    'covered_pass': cov_pass,
    'pass_fraction_among_covered': pass_fraction,
    'time_under_alert': selected_metrics['time_under_alert'],
    'high_origins': high_origins,
    'false_high_event_origins': false_high_events,
    'high_sparse_or_undefined_origins': high_sparse_origins,
    'event_precision': selected_metrics['event_precision'],
    'event_recall': selected_metrics['event_recall']
}])
print('=== PRIMARY METRICS & AUDIT DIAGNOSTICS ===')
print(primary_summary.to_string(index=False))

inc_sel = compute_incidence_matrix(in_span, eval_meta, min_lead=lead_delta)
per_raw_covered = inc_sel[sel_warns.values].any(axis=0) if high_origins > 0 else np.zeros(len(in_span), dtype=bool)
in_span_annot = in_span.copy()
in_span_annot['warned'] = per_raw_covered
in_span_annot['date'] = in_span_annot['timestamp'].dt.floor('D')

eval_end_effective = (planned_eval_end - pd.Timedelta(nanoseconds=1)).floor('D')
all_days = pd.date_range(planned_eval_start.floor('D'), eval_end_effective, freq='D')
per_day_rows = []
for d in all_days:
    day_df = in_span_annot[in_span_annot['date'] == d]
    d_n = len(day_df)
    d_fail = int(day_df['fail'].sum()) if d_n > 0 else 0
    d_cov_n = int(day_df['warned'].sum()) if d_n > 0 else 0
    d_cov_fail = int(day_df[day_df['warned']]['fail'].sum()) if d_n > 0 else 0
    d_cov = (d_cov_fail / d_fail) if d_fail > 0 else np.nan
    d_wl = (d_cov_n / d_n) if d_n > 0 else np.nan
    per_day_rows.append({
        'date': str(d.date()),
        'total_n': d_n,
        'total_fail': d_fail,
        'covered_n': d_cov_n,
        'covered_fail': d_cov_fail,
        'coverage': d_cov,
        'workload': d_wl
    })

df_per_day = pd.DataFrame(per_day_rows)
print('\n=== PER-DAY COMPLETE INCIDENCE BREAKDOWN ===')
print(df_per_day.to_string(index=False))

=== PRIMARY METRICS & AUDIT DIAGNOSTICS ===
              policy  covered_fail  total_fail  coverage  covered_n  total_n  workload  covered_pass  pass_fraction_among_covered  time_under_alert  high_origins  false_high_event_origins  high_sparse_or_undefined_origins  event_precision  event_recall
lr_history_c01 @ 0.2             8          16       0.5        112      297  0.377104           104                     0.928571          0.267857             8                         7                                 0            0.125      0.090909

=== PER-DAY COMPLETE INCIDENCE BREAKDOWN ===
      date  total_n  total_fail  covered_n  covered_fail  coverage  workload
2008-10-03       17           0         17             0       NaN  1.000000
2008-10-04       18           1         18             1  1.000000  1.000000
2008-10-05       32           6          6             1  0.166667  0.187500
2008-10-06       48           5         48             5  1.000000  1.000000
2008-10-07       28

In [6]:
# 6. Two-Day Calendar Block Bootstrap Uncertainty
import math

in_span_blocks = in_span_annot.copy()
in_span_blocks['seconds_from_start'] = (in_span_blocks['timestamp'] - planned_eval_start).dt.total_seconds()
in_span_blocks['block_id'] = (in_span_blocks['seconds_from_start'] // (2 * 86400)).astype(int)

eval_duration_sec = (planned_eval_end - planned_eval_start).total_seconds()
n_blocks_span = math.ceil(eval_duration_sec / (2 * 86400))
all_span_blocks = list(range(n_blocks_span))

block_data = {
    b: in_span_blocks[in_span_blocks['block_id'] == b]
    for b in all_span_blocks
}

n_boot = 1000
rng = np.random.default_rng(42)
boot_coverages = []
boot_workloads = []
valid_replicates = 0

for _ in range(n_boot):
    sampled_b = rng.choice(all_span_blocks, size=n_blocks_span, replace=True)
    resamp_list = [block_data[b] for b in sampled_b if len(block_data[b]) > 0]
    if len(resamp_list) > 0:
        resamp_df = pd.concat(resamp_list, ignore_index=True)
        b_total_fail = int(resamp_df['fail'].sum())
        b_total_n = len(resamp_df)
        b_cov_fail = int(resamp_df[resamp_df['warned']]['fail'].sum())
        b_cov_n = int(resamp_df['warned'].sum())
    else:
        b_total_fail = 0
        b_total_n = 0
        b_cov_fail = 0
        b_cov_n = 0

    b_wl = (b_cov_n / b_total_n) if b_total_n > 0 else np.nan
    boot_workloads.append(b_wl)

    if b_total_fail > 0:
        boot_coverages.append(b_cov_fail / b_total_fail)
        valid_replicates += 1
    else:
        boot_coverages.append(np.nan)

boot_cov_arr = np.array(boot_coverages)
boot_wl_arr = np.array(boot_workloads)

cov_ci = np.nanpercentile(boot_cov_arr, [2.5, 97.5]) if valid_replicates > 0 else (np.nan, np.nan)
wl_ci = np.nanpercentile(boot_wl_arr, [2.5, 97.5])

bootstrap_summary = {
    'n_resamples': n_boot,
    'n_blocks_in_span': n_blocks_span,
    'valid_coverage_replicates': valid_replicates,
    'coverage_median': float(np.nanmedian(boot_cov_arr)) if valid_replicates > 0 else np.nan,
    'coverage_95ci_low': float(cov_ci[0]),
    'coverage_95ci_high': float(cov_ci[1]),
    'workload_median': float(np.nanmedian(boot_wl_arr)),
    'workload_95ci_low': float(wl_ci[0]),
    'workload_95ci_high': float(wl_ci[1]),
    'interpretation': 'Conditional exploratory block bootstrap on frozen model and fixed replay sequence. Does NOT guarantee industrial reproducibility or formal confidence guarantees.'
}
print('=== 2-DAY CALENDAR BLOCK BOOTSTRAP UNCERTAINTY ===')
for k, v in bootstrap_summary.items():
    print(k + ' : ' + str(v))

=== 2-DAY CALENDAR BLOCK BOOTSTRAP UNCERTAINTY ===
n_resamples : 1000
n_blocks_in_span : 7
valid_coverage_replicates : 1000
coverage_median : 0.5
coverage_95ci_low : 0.14285714285714285
coverage_95ci_high : 0.5833333333333334
workload_median : 0.3771043771043771
workload_95ci_low : 0.06966246392949939
workload_95ci_high : 0.6311239193083573
interpretation : Conditional exploratory block bootstrap on frozen model and fixed replay sequence. Does NOT guarantee industrial reproducibility or formal confidence guarantees.


In [7]:
# 7. Prospective 3-Level Replay & Separate Post-Target Outcome Audit
scores_scoring = predict_score(selected_name, deploy_models[selected_name], X_scoring)
init_tau_high = frozen_thresholds[selected_name][selected_budget]
cand_thresholds = frozen_thresholds[selected_name]
init_tau_track = cand_thresholds.get(0.50, np.inf if not isinstance(cand_thresholds, dict) else list(cand_thresholds.values())[-1])

decision_records = []
for orig in eval_origins:
    tau_h = selected_replay['taus'][orig]
    tau_tr_cand, _ = capacity_threshold_at_origin(
        scores=scores_scoring,
        metadata=meta_scoring,
        raw_df=df_raw,
        current_origin=orig,
        model_ready_time=model_ready_time,
        budget=0.50,
        lookback=lookback_delta,
        min_lead=lead_delta,
        n_quantiles=n_quantiles_policy,
        frozen_initial_threshold=init_tau_track
    )
    tau_tr = min(tau_h, tau_tr_cand) if np.isfinite(tau_tr_cand) else tau_h
    sc = scores_scoring.loc[orig]

    if sc >= tau_h:
        level = 'Nguy cơ cao'
        action = 'QC kỹ thuật ưu tiên kiểm tra lại; xác minh hiệu chuẩn đo kiểm'
    elif sc >= tau_tr:
        level = 'Cần theo dõi'
        action = 'Tăng cường theo dõi; ghi nhận xu hướng đo xa'
    else:
        level = 'Bình thường'
        action = 'Quy trình kiểm tra QC tiêu chuẩn định kỳ'

    row_meta = meta_scoring.loc[orig]
    decision_records.append({
        'origin': orig,
        'feature_available_cutoff': row_meta['max_feature_available'],
        'target_start': row_meta['target_start'],
        'target_end': row_meta['target_end'],
        'score': sc,
        'tau_track': tau_tr,
        'tau_high': tau_h,
        'alert_level': level,
        'operational_action': action
    })

df_decisions = pd.DataFrame(decision_records)

# Assertions on df_decisions levels matching score and thresholds
for idx, r in df_decisions.iterrows():
    if r['score'] >= r['tau_high']:
        assert r['alert_level'] == 'Nguy cơ cao', 'Level mismatch for Nguy cơ cao!'
    elif r['score'] >= r['tau_track']:
        assert r['alert_level'] == 'Cần theo dõi', 'Level mismatch for Cần theo dõi!'
    else:
        assert r['alert_level'] == 'Bình thường', 'Level mismatch for Bình thường!'

audit_as_of = planned_eval_end + delay_delta
eval_meta_matured = eval_meta[eval_meta['target_available'] <= audit_as_of]

outcome_audit_records = []
for orig in eval_meta_matured.index:
    row_meta = eval_meta_matured.loc[orig]
    outcome_audit_records.append({
        'origin': orig,
        'target_start': row_meta['target_start'],
        'target_end': row_meta['target_end'],
        'target_available': row_meta['target_available'],
        'event_label_truth': row_meta['event_label'],
        'target_actual_fails': row_meta['n_fail'],
        'target_actual_samples': row_meta['n_future']
    })

df_outcome_audit = pd.DataFrame(outcome_audit_records)

# Post-audit join strictly after separate creation
df_post_audit = pd.merge(df_decisions, df_outcome_audit, on=['origin', 'target_start', 'target_end'], how='inner')

decision_display_cols = ['origin', 'feature_available_cutoff', 'target_start', 'target_end', 'score', 'tau_track', 'tau_high', 'alert_level', 'operational_action']
print('=== PROSPECTIVE DECISION REPLAY TABLE (First 12 Origins) ===')
display(df_decisions[decision_display_cols].head(12))

audit_display_cols = ['origin', 'target_end', 'target_available', 'alert_level', 'event_label_truth', 'target_actual_fails', 'target_actual_samples']
print('\n=== SEPARATE MATURED OUTCOME AUDIT TABLE (First 12 Origins) ===')
display(df_post_audit[audit_display_cols].head(12))

=== PROSPECTIVE DECISION REPLAY TABLE (First 12 Origins) ===


,origin,feature_available_cutoff,target_start,target_end,score,tau_track,tau_high,alert_level,operational_action
0,2008-10-03 00:00:00,2008-10-02 23:19:00,2008-10-03 06:00:00,2008-10-04 06:00:00,0.746286,0.683186,0.743432,Nguy cơ cao,QC kỹ thuật ưu tiên kiểm tra lại; xác minh hiệ...
1,2008-10-03 06:00:00,2008-10-03 05:32:00,2008-10-03 12:00:00,2008-10-04 12:00:00,0.772261,0.690081,0.743432,Nguy cơ cao,QC kỹ thuật ưu tiên kiểm tra lại; xác minh hiệ...
2,2008-10-03 12:00:00,2008-10-03 11:45:00,2008-10-03 18:00:00,2008-10-04 18:00:00,0.788082,0.696944,0.743432,Nguy cơ cao,QC kỹ thuật ưu tiên kiểm tra lại; xác minh hiệ...
3,2008-10-03 18:00:00,2008-10-03 17:40:00,2008-10-04 00:00:00,2008-10-05 00:00:00,0.765299,0.696944,0.743432,Nguy cơ cao,QC kỹ thuật ưu tiên kiểm tra lại; xác minh hiệ...
4,2008-10-04 00:00:00,2008-10-03 18:31:00,2008-10-04 06:00:00,2008-10-05 06:00:00,0.679648,0.696944,0.743432,Bình thường,Quy trình kiểm tra QC tiêu chuẩn định kỳ
5,2008-10-04 06:00:00,2008-10-04 05:46:00,2008-10-04 12:00:00,2008-10-05 12:00:00,0.689552,0.701697,0.743432,Bình thường,Quy trình kiểm tra QC tiêu chuẩn định kỳ
6,2008-10-04 12:00:00,2008-10-04 10:45:00,2008-10-04 18:00:00,2008-10-05 18:00:00,0.672598,0.701697,0.742673,Bình thường,Quy trình kiểm tra QC tiêu chuẩn định kỳ
7,2008-10-04 18:00:00,2008-10-04 15:29:00,2008-10-05 00:00:00,2008-10-06 00:00:00,0.668339,0.701697,0.742673,Bình thường,Quy trình kiểm tra QC tiêu chuẩn định kỳ
8,2008-10-05 00:00:00,2008-10-04 22:58:00,2008-10-05 06:00:00,2008-10-06 06:00:00,0.699100,0.701697,0.742673,Bình thường,Quy trình kiểm tra QC tiêu chuẩn định kỳ
9,2008-10-05 06:00:00,2008-10-05 05:18:00,2008-10-05 12:00:00,2008-10-06 12:00:00,0.752889,0.701697,0.755792,Cần theo dõi,Tăng cường theo dõi; ghi nhận xu hướng đo xa



=== SEPARATE MATURED OUTCOME AUDIT TABLE (First 12 Origins) ===


,origin,target_end,target_available,alert_level,event_label_truth,target_actual_fails,target_actual_samples
0,2008-10-03 00:00:00,2008-10-04 06:00:00,2008-10-04 12:00:00,Nguy cơ cao,0.0,0,23
1,2008-10-03 06:00:00,2008-10-04 12:00:00,2008-10-04 18:00:00,Nguy cơ cao,0.0,0,20
2,2008-10-03 12:00:00,2008-10-04 18:00:00,2008-10-05 00:00:00,Nguy cơ cao,0.0,0,21
3,2008-10-03 18:00:00,2008-10-05 00:00:00,2008-10-05 06:00:00,Nguy cơ cao,0.0,1,18
4,2008-10-04 00:00:00,2008-10-05 06:00:00,2008-10-05 12:00:00,Bình thường,1.0,2,20
5,2008-10-04 06:00:00,2008-10-05 12:00:00,2008-10-05 18:00:00,Bình thường,0.0,2,28
6,2008-10-04 12:00:00,2008-10-05 18:00:00,2008-10-06 00:00:00,Bình thường,1.0,6,35
7,2008-10-04 18:00:00,2008-10-06 00:00:00,2008-10-06 06:00:00,Bình thường,1.0,6,32
8,2008-10-05 00:00:00,2008-10-06 06:00:00,2008-10-06 12:00:00,Bình thường,1.0,7,36
9,2008-10-05 06:00:00,2008-10-06 12:00:00,2008-10-06 18:00:00,Cần theo dõi,1.0,7,40


In [8]:
# 8. Independent Assertion Verification Suite
print('Running independent test suite...')

# Test 1: Recompute coverage union independently using compute_incidence_matrix
recomp_cov = per_raw_covered.sum()
recomp_fail = in_span_annot[in_span_annot['warned']]['fail'].sum()
assert recomp_cov == selected_metrics['covered_n'], 'Independent covered_n mismatch!'
assert recomp_fail == selected_metrics['covered_fail'], 'Independent covered_fail mismatch!'

# Test 2: Unique raw IDs
assert in_span['raw_row_id'].is_unique, 'df_raw raw_row_id is not unique!'
assert in_span_annot['raw_row_id'].is_unique, 'Annotated df raw_row_id is not unique!'

# Test 3: Denominators invariant across all benchmarked candidates & budgets
for res in benchmark_results:
    assert res['total_fail'] == selected_metrics['total_fail'], 'Total fail denominator altered across configs!'
    assert res['total_n'] == selected_metrics['total_n'], 'Total n denominator altered across configs!'

# Test 4: Chronology and lead time assertions
assert (eval_meta['target_start'] - eval_meta['origin'] >= lead_delta).all(), 'Lead time violated!'
assert (eval_meta['target_end'] > eval_meta['target_start']).all(), 'Target start/end inversion!'
assert (eval_meta['target_available'] >= eval_meta['target_end'] + delay_delta).all(), 'Delay time violated!'

print('All independent assertion checks passed successfully.')

Running independent test suite...
All independent assertion checks passed successfully.


In [9]:
# 9. Acceptance Criteria Verification, Limitations & Artifact Preservation
cov_val = selected_metrics['coverage']
wl_val = selected_metrics['workload']
cov_target_met = (cov_val >= 0.70) if pd.notna(cov_val) else False
wl_target_met = (wl_val <= 0.30) if pd.notna(wl_val) else False
numeric_targets_met = cov_target_met and wl_target_met
production_ready = False

print('=== FINAL ACCEPTANCE GATE EVALUATION ===')
print('Selected Policy       : ' + selected_name + ' @ nominal budget ' + str(selected_budget))
print('Measured Coverage     :', f'{cov_val:.4f}' if pd.notna(cov_val) else 'NaN', '(Target: >= 0.70) ->', ('PASS' if cov_target_met else 'FAIL'))
print('Measured Workload     :', f'{wl_val:.4f}' if pd.notna(wl_val) else 'NaN', '(Target: <= 0.30) ->', ('PASS' if wl_target_met else 'FAIL'))
print('Primary Targets Met   :', ('Primary numeric targets met, pending operational validation' if numeric_targets_met else 'Primary numeric targets NOT MET'))
print('Production Readiness  :', ('Production Ready' if production_ready else 'NOT Production Ready (short-term evaluation, historical holdout exposure, unstable fold-wise performance)'))

print('\n=== CRITICAL SCIENTIFIC LIMITATIONS & REPRODUCIBILITY CONSTRAINTS ===')
print('- Historical Exposure: 314 records evaluated are known from earlier explorations; not completely blind.')
print('- Imbalance & Size   : Limited total Fail events across entire dataset (104 Fails / 1567 records).')
print('- Fixed Assumptions  : 6h QC measurement delay assumed constant; timestamps have no explicit timezones.')
print('- No Machine IDs     : Data contains no machine, tool, chamber or shift telemetry.')
print('- Industrial Readiness: Results do not indicate production readiness; cannot substitute process engineering.')

causal_checks = {
    'hashes_verified': bool(h_secom == exp['file_hashes']['secom_data_sha256'] and h_labels == exp['file_hashes']['secom_labels_sha256']),
    'unique_union_verified': bool(in_span_annot['raw_row_id'].is_unique and recomp_cov == selected_metrics['covered_n']),
    'maturity_verified': bool((eval_meta_matured['target_available'] <= audit_as_of).all()),
    'denominators_fixed': bool(all(res['total_fail'] == selected_metrics['total_fail'] and res['total_n'] == selected_metrics['total_n'] for res in benchmark_results)),
    'frozen_bounds': bool(len(eval_origins) > 0 and (eval_meta['target_start'] >= planned_eval_start).all() and (eval_meta['target_end'] <= planned_eval_end).all())
}

final_eval_dict = {
    'selected_policy': {
        'candidate': selected_name,
        'budget': selected_budget,
        'metrics': selected_metrics,
        'diagnostics': {
            'covered_pass': cov_pass,
            'pass_fraction_among_covered': pass_fraction,
            'high_origins': high_origins,
            'false_high_events': false_high_events,
            'high_sparse_origins': high_sparse_origins
        }
    },
    'benchmarks': df_comparison.to_dict(orient='records'),
    'bootstrap_summary': bootstrap_summary,
    'per_day_incidence': df_per_day.to_dict(orient='records'),
    'decisions_full': df_decisions.to_dict(orient='records'),
    'matured_outcome_audit_full': df_outcome_audit.to_dict(orient='records'),
    'post_audit_full': df_post_audit.to_dict(orient='records'),
    'causal_checks': causal_checks,
    'temporal_bounds': {
        'model_ready_time': str(model_ready_time),
        'policy_ready_time': str(policy_ready_time),
        'earliest_holdout_origin': str(earliest_holdout_origin),
        'planned_eval_start': str(planned_eval_start),
        'planned_eval_end': str(planned_eval_end),
        'audit_as_of': str(audit_as_of)
    },
    'acceptance_gates': {
        'coverage_target_met': bool(cov_target_met),
        'workload_target_met': bool(wl_target_met),
        'numeric_targets_met': bool(numeric_targets_met),
        'production_ready': bool(production_ready)
    },
    'heldout_read_at': str(pd.Timestamp.now()),
    'audit_warning': 'Retraining 03_Train.ipynb resets experiment.joblib and invalidates this evaluation.'
}

exp['final_evaluation'] = final_eval_dict
joblib.dump(exp, 'experiment.joblib')
print('\nUpdated experiment.joblib saved with complete final_evaluation artifact.')

=== FINAL ACCEPTANCE GATE EVALUATION ===
Selected Policy       : lr_history_c01 @ nominal budget 0.2
Measured Coverage     : 0.5000 (Target: >= 0.70) -> FAIL
Measured Workload     : 0.3771 (Target: <= 0.30) -> FAIL
Primary Targets Met   : Primary numeric targets NOT MET
Production Readiness  : NOT Production Ready (short-term evaluation, historical holdout exposure, unstable fold-wise performance)

=== CRITICAL SCIENTIFIC LIMITATIONS & REPRODUCIBILITY CONSTRAINTS ===
- Historical Exposure: 314 records evaluated are known from earlier explorations; not completely blind.
- Imbalance & Size   : Limited total Fail events across entire dataset (104 Fails / 1567 records).
- Fixed Assumptions  : 6h QC measurement delay assumed constant; timestamps have no explicit timezones.
- No Machine IDs     : Data contains no machine, tool, chamber or shift telemetry.
- Industrial Readiness: Results do not indicate production readiness; cannot substitute process engineering.

Updated experiment.joblib sa